In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Setup ──────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid")

intakes_df = pd.read_csv(r"D:\Downloads\austin_intakes.csv")
outcomes_df = pd.read_csv(r"D:\Downloads\austin_outcomes.csv")

# ── Parsing ────────────────────────────────────────────────────────────────────
date_cols_intakes = ["source_date", "date_of_birth", "timestamp"]
date_cols_outcomes = ["outcome_date", "intake_date", "date_of_birth", "timestamp"]

for col in date_cols_intakes:
    intakes_df[col] = pd.to_datetime(intakes_df[col], errors="coerce")

for col in date_cols_outcomes:
    outcomes_df[col] = pd.to_datetime(outcomes_df[col], errors="coerce")

outcomes_df["days_in_shelter"] = pd.to_numeric(outcomes_df["days_in_shelter"], errors="coerce")

# Derive age at intake in years
intakes_df["age_at_intake"] = (
    intakes_df["source_date"] - intakes_df["date_of_birth"]
).dt.days / 365.25

intakes_df["intake_year"] = intakes_df["source_date"].dt.year
intakes_df["type_grouped"] = intakes_df["type"].apply(
    lambda x: x if x in ["Dog", "Cat"] else "Other"
)

# ── INTAKES ────────────────────────────────────────────────────────────────────

# 1. Animal type proportions
fig, ax = plt.subplots()
intakes_df["type_grouped"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_title("Intake Proportions: Dogs vs Cats vs Other")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_01_type_proportions.png")
plt.close()

# 2. Top breeds by type
for animal_type in ["Dog", "Cat"]:
    subset = intakes_df[intakes_df["type"] == animal_type]
    top_breeds = (
        subset["primary_breed"]
        .str.strip()
        .str.title()
        .value_counts()
        .head(15)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    top_breeds.sort_values().plot.barh(ax=ax)
    ax.set_title(f"Top 15 {animal_type} Breeds at Intake")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.savefig(f"D:\\Downloads\\eda_02_top_breeds_{animal_type.lower()}.png")
    plt.close()

# 3. Health condition at intake by animal type
health_by_type = (
    intakes_df.groupby(["type_grouped", "intake_health_condition"])
    .size()
    .reset_index(name="count")
)
health_pivot = health_by_type.pivot(
    index="intake_health_condition", columns="type_grouped", values="count"
).fillna(0)
health_pivot_pct = health_pivot.div(health_pivot.sum(axis=0), axis=1) * 100

fig, ax = plt.subplots(figsize=(10, 6))
health_pivot_pct.plot.bar(ax=ax)
ax.set_title("Health Condition at Intake by Animal Type (%)")
ax.set_xlabel("Health Condition")
ax.set_ylabel("% of Intakes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_03_health_condition.png")
plt.close()

# 4. Spay/neuter status
fig, ax = plt.subplots()
intakes_df["ispreviouslyspayedneutered"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_title("Spayed/Neutered at Intake")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_04_spay_neuter.png")
plt.close()

# 5. Age trends by animal type
fig, ax = plt.subplots(figsize=(10, 6))
for animal_type, group in intakes_df.groupby("type_grouped"):
    group["age_at_intake"].dropna().plot.kde(ax=ax, label=animal_type)
ax.set_title("Age at Intake Distribution by Animal Type")
ax.set_xlabel("Age (years)")
ax.set_xlim(0, 20)
ax.legend()
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_05_age_distribution.png")
plt.close()

# 6. Primary color frequency
top_colors = intakes_df["primary_color"].str.strip().str.title().value_counts().head(20)
fig, ax = plt.subplots(figsize=(10, 6))
top_colors.sort_values().plot.barh(ax=ax)
ax.set_title("Top 20 Primary Colors at Intake")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_06_colors.png")
plt.close()

# 7. Intake trends over the years
yearly = intakes_df.groupby(["intake_year", "type_grouped"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(12, 6))
yearly.plot(ax=ax, marker="o")
ax.set_title("Intake Trends Over Time by Animal Type")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Intakes")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_07_yearly_trends.png")
plt.close()

# 8. Intake source analysis
top_sources = intakes_df["source_name"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
top_sources.sort_values().plot.barh(ax=ax)
ax.set_title("Top 15 Intake Sources")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_08_intake_sources.png")
plt.close()

# Stray vs surrender breakdown by type
stray_surrender = intakes_df[
    intakes_df["source_name"].str.contains("Stray|Surrender", case=False, na=False)
]
stray_surrender_breakdown = (
    stray_surrender.groupby(["source_name", "type_grouped"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=stray_surrender_breakdown, x="count", y="source_name", hue="type_grouped", ax=ax)
ax.set_title("Stray vs Surrender Breakdown by Animal Type")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_08b_stray_surrender.png")
plt.close()

# ── OUTCOMES ───────────────────────────────────────────────────────────────────

# 9. Outcome status proportions
fig, ax = plt.subplots(figsize=(8, 8))
outcomes_df["outcome_status"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_title("Outcome Status Proportions")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_09_outcome_status.png")
plt.close()

# 10. Adoption and euthanasia by breed
outcome_filters = {
    "Adoption": outcomes_df["outcome_status"].str.contains("Adopted", case=False, na=False),
    "Euthanasia": outcomes_df["outcome_status"].str.contains("Euthanized", case=False, na=False)
}

for outcome_label, mask in outcome_filters.items():
    subset = outcomes_df[mask]
    top = subset["primary_breed"].str.strip().str.title().value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    top.sort_values().plot.barh(ax=ax)
    ax.set_title(f"Top 15 Breeds: {outcome_label}")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.savefig(f"D:\\Downloads\\eda_10_{outcome_label.lower()}_by_breed.png")
    plt.close()

# 11. Days in shelter distribution
fig, ax = plt.subplots(figsize=(10, 6))
outcomes_df["days_in_shelter"].clip(upper=365).plot.hist(bins=50, ax=ax)
ax.set_title("Days in Shelter Distribution (capped at 365)")
ax.set_xlabel("Days")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_11_days_in_shelter.png")
plt.close()

print(f"Median days in shelter: {outcomes_df['days_in_shelter'].median():.1f}")
print(f"Mean days in shelter: {outcomes_df['days_in_shelter'].mean():.1f}")

# 12. Euthanasia reasons
euth = outcomes_df[outcomes_df["outcome_status"].str.contains("Euthanized", case=False, na=False)]
euth_reasons = euth["euthanasia_reason"].value_counts()
fig, ax = plt.subplots(figsize=(10, 6))
euth_reasons.sort_values().plot.barh(ax=ax)
ax.set_title("Euthanasia Reasons")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig(r"D:\Downloads\eda_12_euthanasia_reasons.png")
plt.close()

print("EDA complete. Charts saved to D:\\Downloads\\")

Median days in shelter: 202.0
Mean days in shelter: 194.1
EDA complete. Charts saved to D:\Downloads\


In [12]:
counts = intakes_df["animal_id"].value_counts()
duplicates = counts[counts > 1]
duplicates

animal_id
10869    10
7507     10
17740     8
17747     8
11656     8
         ..
5978      2
5948      2
6035      2
6033      2
6032      2
Name: count, Length: 10840, dtype: int64

In [14]:
print(len(intakes_df))
print(len(outcomes_df))

23542
21890
